In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import linear_model, feature_selection, metrics
from sklearn import model_selection, pipeline, preprocessing

# A deep-dive into SGDClassifier

## Load the MNIST data

In [ ]:
DATA_DIR = pathlib.Path("/kaggle/input/digit-recognizer")

train_df = pd.read_csv(DATA_DIR / "train.csv")
train_features_df = train_df.drop("label", axis=1)
train_labels = train_df.loc[:, "label"]

## Data preparation

In [ ]:
features_preparation_pipeline = pipeline.make_pipeline(
    feature_selection.VarianceThreshold(threshold=0.0),
    preprocessing.MinMaxScaler(feature_range=(-1, 1)),
)

In [ ]:
features_preparation_pipeline

## Default SGDClassifier settings

By default, [SGDClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) trains a linear classifier using Stochastic Gradient Descent (SGD), optimized for large-scale and sparse datasets. The defaults configure it to behave like a [linear Support Vector Machine (SVM)](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html) classifier with hinge loss and L2 regularization, trained until convergence (or max iterations).

In [ ]:
sgd_classifier = linear_model.SGDClassifier(
    loss="hinge",
    penalty="l2",
    alpha=0.0001,
    l1_ratio=0.15,
    fit_intercept=True,
    max_iter=1000,
    tol=0.001,
    shuffle=True,
    verbose=0,
    epsilon=0.1,
    n_jobs=None,
    random_state=None,
    learning_rate='optimal',
    eta0=0.0,
    power_t=0.5,
    early_stopping=False,
    validation_fraction=0.1,
    n_iter_no_change=5,
    class_weight=None,
    warm_start=False,
    average=False,
)

In [ ]:
sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    sgd_classifier,
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier_pipeline.fit(
    train_features_df,
    train_labels
)

In [ ]:
predicted_train_labels = sgd_classifier_pipeline.predict(
    train_features_df
)

train_accuracy = metrics.accuracy_score(
    train_labels,
    predicted_train_labels,
)

print(f"Accuracy on the training set: {train_accuracy}")

cv_accuracy_scores = model_selection.cross_val_score(
    sgd_classifier_pipeline,
    train_features_df,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=0,
)

print(f"CV Accuracy: {cv_accuracy_scores.mean()}")

## Adding an intercept

* `fit_intercept=True`: An intercept feature is included in the model.

**Keep `fit_intercept=True` unless both your features and your target are centered around zero!**

In [ ]:
sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
    )
)

## Class weights

* `class_weight=None`: Parameter controls how much importance each class has in the loss function. By default, all classes are treated equally (no class weighting).
* `class_weight="balanced"`: gives the classes weights that are inversely proportionaly to their frequency. This makes rare classes get larger weights, balancing their influence.
* You can also manually give classes weights by passing a dictionary mapping the class labels to their weights.

### When to use?

* With imbalanced classes (e.g. fraud detection, medical diagnosis) "balanced" is a good first try.
* Custom weights if domain knowledge tells you one class is much more important.

### When not to use?

* With symmetric tasks (roughly equal class sizes), keep `class_weight=None`.
* `class_weight="balanced"` can sometimes hurt if classes are already balanced.

In [ ]:
sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight="balanced",
    )
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier_pipeline.fit(
    train_features_df,
    train_labels
)

In [ ]:
predicted_train_labels = sgd_classifier_pipeline.predict(
    train_features_df
)

train_accuracy = metrics.accuracy_score(
    train_labels,
    predicted_train_labels,
)

print(f"Accuracy on the training set: {train_accuracy}")

cv_accuracy_scores = model_selection.cross_val_score(
    sgd_classifier_pipeline,
    train_features_df,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=0,
)

print(f"CV Accuracy: {cv_accuracy_scores.mean()}")

## Parallelism and reproducibility

* `n_jobs=None`: By default, training runs on a single core (or default system settings). Set to an integer to control the number of cores directly, or set to `-1` to use all available cores.
* `random_state=None`: Randomness (e.g., shuffling, weight initialization) is not fixed, so results may vary run to run unless you set a seed.

In [ ]:
prng = np.random.RandomState(42)

sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight=None,
        n_jobs=-1,
        random_state=prng
    )
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier_pipeline.fit(train_features_df, train_labels)

## Choosing the loss function

### `loss="hinge"`

* Trains a [linear Support Vector Machine (SVM)](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html) using SGD.
* Use when you care about margins and separating hyperplanes.
* Good for high-dimensional sparse data (e.g. text classification).
* Doesn’t provide probabilities.

In [ ]:
prng = np.random.RandomState(42)

sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight=None,
        n_jobs=-1,
        random_state=prng,
        loss="hinge",
    )
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier_pipeline.fit(
    train_features_df,
    train_labels
)

In [ ]:
predicted_train_labels = sgd_classifier_pipeline.predict(
    train_features_df
)

train_accuracy = metrics.accuracy_score(
    train_labels,
    predicted_train_labels,
)

print(f"Accuracy on the training set: {train_accuracy}")

cv_accuracy_scores = model_selection.cross_val_score(
    sgd_classifier_pipeline,
    train_features_df,
    train_labels,
    
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=0,
)

print(f"CV Accuracy: {cv_accuracy_scores.mean()}")

### `loss="log_loss"`

* Trains logistic/softmax regression.
* Use when you want probabilities, interpretability, or threshold tuning.
* Often better with imbalanced data, since probabilities can be calibrated.


In [ ]:
prng = np.random.RandomState(42)

sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight=None,
        n_jobs=-1,
        random_state=prng,
        loss="log_loss",
    )
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier_pipeline.fit(
    train_features_df,
    train_labels
)

In [ ]:
predicted_train_labels = sgd_classifier_pipeline.predict(
    train_features_df
)

train_accuracy = metrics.accuracy_score(
    train_labels,
    predicted_train_labels,
)

print(f"Accuracy on the training set: {train_accuracy}")

cv_accuracy_scores = model_selection.cross_val_score(
    sgd_classifier_pipeline,
    train_features_df,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=0,
)

print(f"CV Accuracy: {cv_accuracy_scores.mean()}")

### loss="modified_huber"

* A smooth variant of hinge loss.
* Like `hinge` this loss is robust to outliers.
* Gives **probability-like** scores (but not calibrated probabilities).
* Nice compromise between `hinge` and `log_loss`.

#### Exercise:

Modify the code above to train and evaluate a model with the modified Huber loss function.

In [ ]:
# INSERT YOUR CODE HERE!

## Controlling training iterations

* `max_iter=1000`: The maximum number of passes over the training data (epochs).
* `tol=0.001`: Training stops early if the objective does not improve by at least `tol` for `n_iter_no_change` consecutive epochs.
* `shuffle=True`: Data is shuffled after each epoch to avoid cycles and improve convergence.
* `verbose=0`: No progress messages are printed during training. Values larger than 0 will print more detailed information about training progress.

In [ ]:
prng = np.random.RandomState(42)

sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight=None,
        n_jobs=-1,
        random_state=prng,
        loss="hinge",
        max_iter=1000,
        n_iter_no_change=5,
        shuffle=True,
        tol=0.001,
        verbose=0
    )
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier_pipeline.fit(
    train_features_df,
    train_labels
)

In [ ]:
predicted_train_labels = sgd_classifier_pipeline.predict(
    train_features_df
)

train_accuracy = metrics.accuracy_score(
    train_labels,
    predicted_train_labels,
)

print(f"Accuracy on the training set: {train_accuracy}")

cv_accuracy_scores = model_selection.cross_val_score(
    sgd_classifier_pipeline,
    train_features_df,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=0,
)

print(f"CV Accuracy: {cv_accuracy_scores.mean()}")

## Regularization

Regularization helps improve models that are overfitting the training data. There are two general strategies for regularizing models.

1. **Early-stopping**: reduces over-fitting to model iterations.
2. **Penalizing model parameters**: reduces over-fitting to model features.   

### Avoid overtraining with `early_stopping=True`

* `early_stopping=False`: By default, SGDClassifier does not reserve validation data or stop early based on validation score. Training runs until convergence or max epochs.
* `validation_fraction=0.1`: Fraction of data reserved for validation when `early_stopping=True`. By default `early_stopping=False` and this is ignored.
* `n_iter_no_change=5`: Training will stop if the validation score (or training score, if no validation split) does not improve for `n_iter_no_change` iterations in a row.

When `early_stopping=True`, the training data is split into a training and validation set. After each epoch, performance is checked on the validation set. If the score doesn’t improve for 1n_iter_no_change1 epochs by at least `tol`, training stops early. This helps avoid training too long (a form of overfitting), especially when `max_iter` is large.

In [ ]:
prng = np.random.RandomState(42)

sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight=None,
        n_jobs=-1,
        random_state=prng,
        loss="hinge",
        max_iter=1000,
        n_iter_no_change=5,
        shuffle=True,
        tol=0.001,
        verbose=0,
        early_stopping=True,
        validation_fraction=0.10,
        penalty=None,
    )
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier_pipeline.fit(
    train_features_df,
    train_labels
)

In [ ]:
predicted_train_labels = sgd_classifier_pipeline.predict(
    train_features_df
)

train_accuracy = metrics.accuracy_score(
    train_labels,
    predicted_train_labels,
)

print(f"Accuracy on the training set: {train_accuracy}")

cv_accuracy_scores = model_selection.cross_val_score(
    sgd_classifier_pipeline,
    train_features_df,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=0,
)

print(f"CV Accuracy: {cv_accuracy_scores.mean()}")

### Penalizing model parameters

* `penalty="l2"`: L2 regularization is applied to prevent overfitting. This penalizes large weights by their squared magnitude.
* `alpha=0.0001`: Regularization strength (λ in optimization). Higher values increase regularization (smaller coefficients), lower values allow more complex models.
* `l1_ratio=0.15`: Controls the L1/L2 mix when using `penalty="elasticnet"`. Since the default is `l2`, this value has no effect unless you change penalty.

In [ ]:
prng = np.random.RandomState(42)

sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight=None,
        n_jobs=-1,
        random_state=prng,
        loss="hinge",
        max_iter=1000,
        n_iter_no_change=5,
        shuffle=True,
        tol=0.001,
        verbose=0,
        penalty="l2",
        alpha=0.0001,
        l1_ratio=0.15,
    )
)

#### Exercise:

Modify the code above to train and evaluate a model using an ElasticNet penalty. Can you find a value for the `l1_ratio` hyperparameter to mimic an `l2` penalty? Can you find a value for the `l1_ratio` that mimics an `l1` penalty?   

In [ ]:
# INSERT YOUR CODE HERE!

### Combine both types of regularization

For the most part you will want to always use early stopping in combination with explicit regularization via penalties.

In [ ]:
prng = np.random.RandomState(42)

sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight=None,
        n_jobs=-1,
        random_state=prng,
        loss="hinge",
        max_iter=1000,
        n_iter_no_change=5,
        shuffle=True,
        tol=0.001,
        verbose=0,
        early_stopping=True,
        validation_fraction=0.10,
        penalty="l2",
        alpha=0.0001,
        l1_ratio=0.15,
    )
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier_pipeline.fit(
    train_features_df,
    train_labels
)

In [ ]:
predicted_train_labels = sgd_classifier_pipeline.predict(
    train_features_df
)

train_accuracy = metrics.accuracy_score(
    train_labels,
    predicted_train_labels,
)

print(f"Accuracy on the training set: {train_accuracy}")

cv_accuracy_scores = model_selection.cross_val_score(
    sgd_classifier_pipeline,
    train_features_df,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=0,
)

print(f"CV Accuracy: {cv_accuracy_scores.mean()}")

## Controlling the learning rate

### `learning_rate="optimal"`

Uses a theoretically motivated adaptive learning rate that sets the learning rate as follows

`eta = 1.0 / (alpha * t)` 

where `t` is the iteration count and `alpha` is the amount of parameter regularization.

In [ ]:
alpha = 1e-3
ts = np.linspace(1, 1000, 1000)
_ = plt.plot(ts, 1.0 / (alpha * ts))
_ = plt.title("learning_rate='optimal'")
_ = plt.ylabel("$\eta_t$")
_ = plt.yscale("log")
_ = plt.xlabel("Iteration, $t$")
_ = plt.grid(True)

In [ ]:
prng = np.random.RandomState(42)

sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight=None,
        n_jobs=-1,
        random_state=prng,
        loss="hinge",
        max_iter=1000,
        n_iter_no_change=5,
        shuffle=True,
        tol=0.001,
        verbose=0,
        early_stopping=True,
        validation_fraction=0.10,
        penalty="l2",
        alpha=0.0001,
        l1_ratio=0.15,
        learning_rate="optimal",
        eta0=1e-3,
        power_t=0.5,
    )
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier_pipeline.fit(
    train_features_df,
    train_labels
)

In [ ]:
predicted_train_labels = sgd_classifier_pipeline.predict(
    train_features_df
)

train_accuracy = metrics.accuracy_score(
    train_labels,
    predicted_train_labels,
)

print(f"Accuracy on the training set: {train_accuracy}")

cv_accuracy_scores = model_selection.cross_val_score(
    sgd_classifier_pipeline,
    train_features_df,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=0,
)

print(f"CV Accuracy: {cv_accuracy_scores.mean()}")

### `learning_rate="constant"`

* `eta0=0.0`: Controls the initial learning rate which is them held constant for the remainder of training.

In [ ]:
eta0 = 1e3
ts = np.linspace(1, 1000, 1000)
_ = plt.plot(ts, eta0 * np.ones(1000))
_ = plt.title("learning_rate='constant'")
_ = plt.ylabel("$\eta_t$")
_ = plt.xlabel("Iteration, $t$")
_ = plt.grid(True)

#### Exercise

Modify the code above to use a constant learning rate strategy with a learning rate of 1e-6. Train and evaluate the model. Repeat the experiment with a learning rate of 1e3. What impact does a constant and very small or a constant and very large learning rate have on model performance?

In [ ]:
# INSERT YOUR CODE HERE!

### `learning_rate="invscaling"`

Uses an adaptive learning rate that sets the learning rate as follows

`eta = eta0 / t**power_t`

Where `eta0` is the initial learning rate and `power_t` controls the decay of the learning rate. Values of `power_t` closer to 1 will lead to faster decay of the learning rate; values of `power_t` closer to 0 will lead to slower decay of the learning rate.

In [ ]:
eta0 = 1e3
power_t = 0.5
ts = np.linspace(1, 1000, 1000)
_ = plt.plot(ts, eta0 / ts**power_t)
_ = plt.title("learning_rate='invscaling'")
_ = plt.ylabel("$\eta_t$")
_ = plt.yscale("log")
_ = plt.xlabel("Iteration, $t$")
_ = plt.grid(True)

In [ ]:
prng = np.random.RandomState(42)

sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight=None,
        n_jobs=-1,
        random_state=prng,
        loss="hinge",
        max_iter=1000,
        n_iter_no_change=5,
        shuffle=True,
        tol=0.001,
        verbose=0,
        early_stopping=True,
        validation_fraction=0.10,
        penalty="elasticnet",
        alpha=0.0001,
        l1_ratio=0.15,
        learning_rate="invscaling",
        eta0=1e3,
        power_t=0.5,
    )
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier_pipeline.fit(
    train_features_df,
    train_labels
)

In [ ]:
predicted_train_labels = sgd_classifier_pipeline.predict(
    train_features_df
)

train_accuracy = metrics.accuracy_score(
    train_labels,
    predicted_train_labels,
)

print(f"Accuracy on the training set: {train_accuracy}")

cv_accuracy_scores = model_selection.cross_val_score(
    sgd_classifier_pipeline,
    train_features_df,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=0,
)

print(f"CV Accuracy: {cv_accuracy_scores.mean()}")

### `learning_rate="adaptive"`

Start with the initial learning rate `eta0` (default = 0.0, but here you must set it, e.g. `eta0=0.01` — otherwise training won’t move).

* After each epoch, check if the objective (loss) improves.
* If improvement is observed, then keep the same learning rate.
* If no improvement for `n_iter_no_change` consecutive epochs, then divide the learning rate by 5.

Setting `learning_rate="adaptive"` adjusts the learning rate in a fashion that is similar to learning rate scheduling in deep learning frameworks:

* Stay aggressive while making progress.
* Automatically slow down if the model stops improving.

Helps balance speed (large steps early) and stability (smaller steps when near a minimum).

In [ ]:
prng = np.random.RandomState(42)

sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight=None,
        n_jobs=-1,
        random_state=prng,
        loss="hinge",
        max_iter=1000,
        n_iter_no_change=5,
        shuffle=True,
        tol=0.001,
        verbose=0,
        early_stopping=True,
        validation_fraction=0.10,
        penalty="elasticnet",
        alpha=0.0001,
        l1_ratio=0.15,
        learning_rate="adaptive",
        eta0=1e-2,
    )
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier_pipeline.fit(
    train_features_df,
    train_labels
)

In [ ]:
predicted_train_labels = sgd_classifier_pipeline.predict(
    train_features_df
)

train_accuracy = metrics.accuracy_score(
    train_labels,
    predicted_train_labels,
)

print(f"Accuracy on the training set: {train_accuracy}")

cv_accuracy_scores = model_selection.cross_val_score(
    sgd_classifier_pipeline,
    train_features_df,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=0,
)

print(f"CV Accuracy: {cv_accuracy_scores.mean()}")

## Averaging parameters

The `average` hyperparameter in `SGDClassifier` controls whether the model uses **Polyak–Ruppert averaging** of the parameters during training. This is a technique from stochastic optimization that can improve stability and generalization.

### What normally happens (`average=False`)

* In vanilla SGD, parameters are updated after each sample (or mini-batch).
* At the end of training, the model keeps the **last parameters**.

### What happens with `average=True`

* Instead of just keeping the final parameters, the model computes the **running average of all past weights**:

$$ \bar{w}_T = \frac{1}{T}\sum_{t=1}^T w_t $$

* This smoothed weight vector is used for predictions.
* The idea: averaging cancels out the “noise” of individual updates, leading to more stable convergence.

### If you set `average=int`

* The model will **start averaging only after that many updates**.
* Example: `average=1000` → first 1000 updates are standard SGD, after that averaging kicks in.
* This is useful as a **warm-up** so the model can move quickly at the start, then stabilize.

### Why averaging helps

SGD updates are noisy because each step is based on a single (or small batch) sample. Averaging parameters:

* Reduces variance in the parameters.
* Improves generalization (especially in noisy, high-dimensional, or sparse problems).
* It’s particularly effective for **linear models** trained with SGD.

In [ ]:
prng = np.random.RandomState(42)

sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight=None,
        n_jobs=-1,
        random_state=prng,
        loss="hinge",
        max_iter=1000,
        n_iter_no_change=5,
        shuffle=True,
        tol=0.001,
        verbose=0,
        early_stopping=True,
        validation_fraction=0.10,
        penalty="elasticnet",
        alpha=0.0001,
        l1_ratio=0.15,
        learning_rate="adaptive",
        eta0=1e-2,
        average=42_000 * 3, # only start averaging after this many updates (~3 epochs!)
    )
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier_pipeline.fit(
    train_features_df,
    train_labels
)

In [ ]:
predicted_train_labels = sgd_classifier_pipeline.predict(
    train_features_df
)

train_accuracy = metrics.accuracy_score(
    train_labels,
    predicted_train_labels,
)

print(f"Accuracy on the training set: {train_accuracy}")

cv_accuracy_scores = model_selection.cross_val_score(
    sgd_classifier_pipeline,
    train_features_df,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=0,
)

print(f"CV Accuracy: {cv_accuracy_scores.mean()}")

## Hyperparameter tuning strategy

### Step 1: Choose your loss function

Choose the loss function most appropriate for your problem and set up early stopping. In addition to helping regularize our models, early stopping will significantly speed up hyperparameter tuning runs.

In [ ]:
prng = np.random.RandomState(42)

sgd_classifier_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        fit_intercept=True,
        class_weight=None,
        n_jobs=-1,
        random_state=prng,
        loss="hinge",
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=5,
        tol=0.001,
    )
)

In [ ]:
param_grid = [
    {
        "sgdclassifier__loss": [
            "hinge",
            "log_loss",
            "modified_huber",
            "squared_hinge",
            "perceptron",
        ],
    }
]

loss_function_tuning = model_selection.GridSearchCV(
    sgd_classifier_pipeline,
    param_grid,
    cv=4,
    n_jobs=-1,
    refit=True,
    scoring="accuracy",
    verbose=0,
)

In [ ]:
loss_function_tuning

In [ ]:
_ = loss_function_tuning.fit(
    train_features_df,
    train_labels,
)

In [ ]:
loss_function_tuning.best_params_

In [ ]:
loss_function_tuning.best_score_

In [ ]:
(
    pd.DataFrame
      .from_dict(
          loss_function_tuning.cv_results_
      ).sort_values(
          by="rank_test_score"
      )
)

### Step 2: Tune regularization hyperparameters

In [ ]:
param_grid = [
    {
        "sgdclassifier__penalty": ["elasticnet"],
        "sgdclassifier__alpha": np.logspace(-6, -1, 6),
        "sgdclassifier__l1_ratio": [0, 0.15, 0.3, 0.45, 0.6, 0.85, 1.0],
        "sgdclassifier__n_iter_no_change": [5, 10, 15]
    }
]

regularization_tuning = model_selection.RandomizedSearchCV(
    loss_function_tuning.best_estimator_,
    param_grid,
    cv=4,
    n_iter=25,
    n_jobs=-1,
    refit=True,
    scoring="accuracy",
    verbose=0,
)

In [ ]:
regularization_tuning

In [ ]:
_ = regularization_tuning.fit(
    train_features_df,
    train_labels
)

In [ ]:
# accessing the best hyperparameters
regularization_tuning.best_params_

In [ ]:
# accessing the score for the best hyperparameters
regularization_tuning.best_score_

In [ ]:
(
    pd.DataFrame
      .from_dict(
          regularization_tuning.cv_results_
      ).sort_values(
          by="rank_test_score"
      )
)

### Step 3: Tune the Learning Rate

In [ ]:
param_distributions = [
    {
        "sgdclassifier__learning_rate": ["optimal"],
    },
    {
        "sgdclassifier__learning_rate": ["adaptive"],
        "sgdclassifier__eta0": [1e-3, 1e-2, 1e0, 1e2, 1e3],
        "sgdclassifier__n_iter_no_change": [5, 10, 15, 20]
    },
    {
        "sgdclassifier__learning_rate": ["invscaling"],
        "sgdclassifier__eta0": [1e-3, 1e-2, 1e0, 1e2, 1e3],
        "sgdclassifier__power_t": [0.25, 0.5, 0.75],
    }
]

learning_rate_tuning = model_selection.RandomizedSearchCV(
    regularization_tuning.best_estimator_,
    param_distributions,
    cv=4,
    n_jobs=-1,
    n_iter=25,
    refit=True,
    scoring="accuracy",
    verbose=0,
)

In [ ]:
_ = learning_rate_tuning.fit(
    train_features_df,
    train_labels
)

In [ ]:
learning_rate_tuning.best_params_

In [ ]:
learning_rate_tuning.best_score_

In [ ]:
(
    pd.DataFrame
      .from_dict(
          learning_rate_tuning.cv_results_
      ).sort_values(
          by="rank_test_score"
      )
)

### Step 4: Try averaging parameters

In [ ]:
param_grid = [
    {
        "sgdclassifier__average": [
            False,
            True,
            42_000 * 5,
            42_000 * 10,
            42_000 * 15
        ],
    },
]

average_tuning = model_selection.GridSearchCV(
    learning_rate_tuning.best_estimator_,
    param_grid,
    cv=4,
    n_jobs=-1,
    refit=True,
    scoring="accuracy",
    verbose=0,
)

In [ ]:
_ = average_tuning.fit(
    train_features_df,
    train_labels
)

In [ ]:
average_tuning.best_params_

In [ ]:
average_tuning.best_score_

In [ ]:
(
    pd.DataFrame
      .from_dict(
          average_tuning.cv_results_
      ).sort_values(
          by="rank_test_score"
      )
)

## Submitting to the competition

### Generate test predictions

In [ ]:
test_features_df = pd.read_csv(DATA_DIR / "test.csv")
test_predictions = average_tuning.predict(test_features_df)


### Create submission file

In [ ]:
_ = (
    pd.read_csv(
        "/kaggle/input/digit-recognizer/sample_submission.csv",
        index_col="ImageId"
    ).assign(
        Label=test_predictions
    ).to_csv("submission.csv", index=True)
)

In [ ]:
%%bash

cat submission.csv | head -n 5